# Build cross-session agent memory with a JSON knowledge graph

By Kübra Altinel Bruun ([@kaltinel](https://github.com/kaltinel))


Give a chat model persistent, inspectable memory by extracting structured facts after each message and retrieving them before the next call.

> This cookbook uses the Chat Completions API with JSON mode for extraction. Appendix A shows the same loop running fully offline with Ollama; Appendix B uses Voxtral TTS. Everything is stored as plain JSON you can inspect and delete.

## Prerequisites

To complete this cookbook, you will need:

- Python 3.10 or later, runnable in Jupyter or Google Colab
- A Mistral account and API key
- Ollama with an open-weight model (optional, Appendix A only)

## Environment setup

Install the packages and set up your API key before running the steps below.

### Install

Install the Mistral Python SDK:

In [1]:
!pip install -q "mistralai>=1.5,<2"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


### Required environment variables

To complete this cookbook, you'll need a Mistral API key. In [Studio](https://console.mistral.ai), navigate to the [API keys section](https://console.mistral.ai/home?profile_dialog=api-keys) and create a new API key.

Set `MISTRAL_API_KEY` as an environment variable before running (local use), or leave it unset and enter the key when the next cell prompts you.

In [2]:
import getpass
import os

if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral API key: ")


Mistral API key: ··········


In [3]:
from mistralai import Mistral

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
MODEL = "mistral-small-latest"   # check https://docs.mistral.ai/getting-started/models/models_overview/ for current model ids


## 1. Show that a chat call keeps no state

Call the model twice in separate requests. The second call has no access to the first, so it cannot answer a question about it and this is the gap the memory layer fills.

In [6]:
def chat(messages, temperature=0.7):
    r = client.chat.complete(model=MODEL, messages=messages, temperature=temperature)
    return r.choices[0].message.content.strip()

# Give your name to introduce information to the model
print("call 1:", chat([{"role": "user", "content": "My name is Kübra."}])[:80])

# Ask for the name
print("call 2 (separate request):", chat([{"role": "user", "content": "What is my name?"}]))


call 1: Merhaba Kübra! 😊 How can I assist you today?
call 2 (separate request): I don’t have access to personal information unless you share it with me. If you tell me your name, I can use it in our conversation! Otherwise, I’ll just refer to you as "you" or "friend." 😊

What’s your name?


## 2. Extract facts as triples with JSON mode

After each user message, a second call at temperature 0 returns a JSON object of `subject -- relation --> object` triples. `response_format={"type": "json_object"}` keeps the output parseable.

In [7]:
import json

EXTRACT_PROMPT = (
    "From the message below, extract durable facts as subject-relation-object triples. "
    'If the speaker is the reader (I, me, my), use the subject name "the user". '
    "Use short snake_case relations like name, works_on, is_about, prefers. "
    'Reply with ONLY a JSON object of the form {"triples": [{"subject": ..., "relation": ..., "object": ...}]}. '
    'Reply {"triples": []} if there are no durable facts.\n\nMessage: '
)

def extract_triples(message):
    r = client.chat.complete(
        model=MODEL, temperature=0,
        messages=[{"role": "user", "content": EXTRACT_PROMPT + message}],
        response_format={"type": "json_object"},
    )
    try:
        data = json.loads(r.choices[0].message.content)
        return [t for t in data.get("triples", []) if isinstance(t, dict)]
    except (json.JSONDecodeError, AttributeError):
        return []

for t in extract_triples("My name is Kübra and my project local_ai_memory is about knowledge graphs."):
    print(t)


{'subject': 'Kübra', 'relation': 'name', 'object': 'Kübra'}
{'subject': 'Kübra', 'relation': 'works_on', 'object': 'local_ai_memory'}
{'subject': 'local_ai_memory', 'relation': 'is_about', 'object': 'knowledge graphs'}


## 3. Store facts in a JSON knowledge graph

Nodes are entities, edges are relations, and the whole graph is a dict you can print, save, and delete. Retrieval is keyword-based: name match, token overlap, first-person pronouns resolve to "the user", recent context resolves "it/this", then a 2-hop expansion.

In [8]:
import re

PRONOUNS = {"i", "me", "my", "mine", "myself"}
STOP = {"a","an","the","is","are","was","am","to","of","in","on","at","and","or","what",
        "who","how","do","does","did","you","your","it","its","this","that","about",
        "tell","know","like","with","for","from","have","has","had","not","be","been","as","by"}
def norm(s): return re.sub(r"\s+", " ", str(s)).strip().lower()
def toks(s): return {t for t in re.findall(r"[a-z0-9_]+", str(s).lower()) if t not in STOP}

class KG:
    def __init__(self):
        self.nodes, self.edges = [], []
    def names(self):
        return {n["id"]: n["name"] for n in self.nodes}
    def find(self, name):
        n = norm(name)
        return next((x for x in self.nodes if norm(x["name"]) == n), None)
    def node(self, name, type="concept", is_user=False):
        n = self.find(name)
        if n:
            return n
        n = {"id": f"n{len(self.nodes)+1}", "type": type, "name": str(name).strip(), "is_user": is_user}
        self.nodes.append(n)
        return n
    def line(self, e):
        ids = self.names()
        return f"{ids[e['subject']]} --{e['relation']}--> {ids[e['object']]}"
    def absorb(self, triples):
        out = []
        for t in triples:
            s, r, o = t.get("subject"), t.get("relation"), t.get("object")
            if not all(isinstance(v, str) and v.strip() for v in (s, r, o)):
                continue
            su = norm(s).replace("_", " ")
            is_user = su in PRONOUNS or su in {"the user", "user"}
            if is_user:
                s = "the user"
            a = self.node(s, "person" if is_user else "concept", is_user)
            b = self.node(o)
            if any(e["subject"] == a["id"] and norm(e["relation"]) == norm(r) and e["object"] == b["id"] for e in self.edges):
                out.append(("updated", f"{a['name']} --{r}--> {b['name']}"))
                continue
            self.edges.append({"subject": a["id"], "relation": r, "object": b["id"]})
            out.append(("added", f"{a['name']} --{r}--> {b['name']}"))
        return out
    def _seeds(self, text):
        tk, low = toks(text), str(text).lower()
        scored = []
        for n in self.nodes:
            s = (3 if norm(n["name"]) and norm(n["name"]) in low else 0) \
                + len(toks(n["name"]) & tk) \
                + (5 if n["is_user"] and toks(text) & PRONOUNS else 0)
            if s:
                scored.append((s, n))
        return [n for _, n in sorted(scored, key=lambda x: -x[0])[:5]]
    def lines(self, text, context="", hops=2):
        seeds = self._seeds(f"{text} {context}")
        if not seeds:
            return []
        seen = {n["id"] for n in seeds}
        frontier = set(seen)
        for _ in range(hops):
            nxt = set()
            for e in self.edges:
                if e["subject"] in frontier:
                    nxt.add(e["object"])
                if e["object"] in frontier:
                    nxt.add(e["subject"])
            frontier = nxt - seen
            seen |= frontier
        return [self.line(e) for e in self.edges if e["subject"] in seen or e["object"] in seen]
    def conflicts(self):
        g = {}
        for e in self.edges:
            g.setdefault((e["subject"], norm(e["relation"])), []).append(e)
        out = []
        for (s, r), es in g.items():
            if len({e["object"] for e in es}) > 1:
                out.append((self.names()[s], r, [self.line(e) for e in es]))
        return out
    def all_lines(self):
        return [self.line(e) for e in self.edges]
    def to_json(self):
        return json.dumps({"nodes": self.nodes, "edges": self.edges}, indent=2)

kg = KG()
print("empty graph created")


empty graph created


## 4. Run the memory loop across sessions

Each turn retrieves the relevant subgraph into the system prompt, calls the model, then extracts and merges triples from the message. A "new session" is just an empty chat history — the graph persists as JSON.

In [9]:
history = []

def turn(message):
    global history
    context = " ".join(m["content"] for m in history[-4:])
    lines = kg.lines(message, context=context)
    system = "You are a helpful assistant. Known facts (subject --relation--> object):\n" \
        + ("\n".join(f"- {l}" for l in lines) if lines else "(none)")
    msgs = [{"role": "system", "content": system}] + history[-8:] + [{"role": "user", "content": message}]
    reply = chat(msgs)
    history += [{"role": "user", "content": message}, {"role": "assistant", "content": reply}]
    for status, line in kg.absorb(extract_triples(message)):
        print(f"  [kg {status}]", line)
    return reply

for m in ["My name is Kübra and I am building a demo about local AI memory.",
          "My project is called local_ai_memory and it is about knowledge graphs for local models.",
          "While coding I prefer coffee over tea."]:
    print("You  :", m)
    print("Model:", turn(m)[:100], "...\n")

print("Graph:")
print("\n".join("  " + l for l in kg.all_lines()))


You  : My name is Kübra and I am building a demo about local AI memory.
  [kg added] Kübra --name--> Kübra
  [kg added] Kübra --works_on--> local AI memory demo
Model: That sounds like an interesting project, Kübra! Local AI memory could involve storing and retrieving ...

You  : My project is called local_ai_memory and it is about knowledge graphs for local models.
  [kg added] the user --name--> local_ai_memory
  [kg added] local_ai_memory --is_about--> knowledge graphs
  [kg added] local_ai_memory --works_on--> local models
Model: That’s a great project idea, Kübra! **Knowledge graphs (KGs)** combined with **local AI models** can ...

You  : While coding I prefer coffee over tea.
  [kg added] the user --prefers--> coffee
  [kg added] the user --prefers--> tea
  [kg added] coffee --is_about--> the user
  [kg added] tea --is_about--> the user
Model: Haha, noted! ☕️ **Coffee > Tea** for coding sessions—totally agree. The extra caffeine helps with fo ...

Graph:
  Kübra --name--> Kübra


Now start a fresh session — empty chat history, only the graph — and ask a two-hop question.

In [10]:
history = []
q = "What project am I working on, and what is it about?"
print("Retrieved subgraph:")
for l in kg.lines(q):
    print("  -", l)
print("Model:", turn(q))


Retrieved subgraph:
  - the user --name--> local_ai_memory
  - local_ai_memory --is_about--> knowledge graphs
  - local_ai_memory --works_on--> local models
  - the user --prefers--> coffee
  - the user --prefers--> tea
  - coffee --is_about--> the user
  - tea --is_about--> the user
  [kg added] the user --works_on--> a project
  [kg added] the project --is_about--> unknown
Model: You're working on **local_ai_memory**, a project about **knowledge graphs** that utilizes **local models**.


## 5. Consolidate offline ("dreaming")

Between sessions, replay the graph: detect conflicting edges (same subject and relation, different objects), let the model pick the correct one, and write a short profile summary.

In [11]:
# create a contradiction: tea first, then coffee
for m in ["While coding I prefer tea.", "Actually I changed my mind: I prefer coffee now."]:
    kg.absorb(extract_triples(m))
print("Conflicts:")
for s, r, opts in kg.conflicts():
    print(f"  {s} --{r}--> " + " | ".join(o.split("-->")[-1].strip() for o in opts))

DREAM_PROMPT = (
    "You consolidate a knowledge graph offline. Edges are subject --relation--> object. "
    "Conflicts (same subject+relation, different objects) are listed below. "
    "Decide which option is correct (later statements win). "
    'Reply ONLY with JSON: {"fixes": [{"remove": "a --r--> b", "keep": "a --r--> c"}], '
    '"summary": "2-3 sentence profile"}\n'
)

def dream(dry_run=True):
    prompt = DREAM_PROMPT + "\nEdges:\n" + "\n".join(kg.all_lines())
    for s, r, opts in kg.conflicts():
        prompt += f"\nConflict: {s} --{r}--> " + " | ".join(o.split("-->")[-1].strip() for o in opts)
    r = client.chat.complete(model=MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"})
    res = json.loads(r.choices[0].message.content)
    for f in res.get("fixes", []):
        for e in list(kg.edges):
            if kg.line(e) == f["remove"]:
                kg.edges.remove(e)
                print(("dry-run, would remove: " if dry_run else "removed: ") + f["remove"])
        if not dry_run and f.get("keep"):
            m2 = re.match(r"^(.*?)\s*--(.*?)-->\s*(.*)$", f["keep"])
            if m2:
                a = kg.node(m2.group(1).strip())
                b = kg.node(m2.group(3).strip())
                kg.edges.append({"subject": a["id"], "relation": m2.group(2).strip(), "object": b["id"]})
    print("summary:", res.get("summary", "")[:160])
    return res

dream(dry_run=True)   # set to False to apply the fixes


Conflicts:
  the user --prefers--> coffee | tea
  the user --works_on--> a project | coding
dry-run, would remove: the user --prefers--> coffee
dry-run, would remove: the user --works_on--> a project
summary: The knowledge graph consolidates user preferences and projects, resolving conflicts by prioritizing later statements. 'Tea' is kept over 'coffee' for user prefe


{'fixes': [{'remove': 'the user --prefers--> coffee',
   'keep': 'the user --prefers--> tea'},
  {'remove': 'the user --works_on--> a project',
   'keep': 'the user --works_on--> coding'}],
 'summary': "The knowledge graph consolidates user preferences and projects, resolving conflicts by prioritizing later statements. 'Tea' is kept over 'coffee' for user preference, and 'coding' is retained over 'a project' for user work. All other edges remain unchanged."}

## 6. Compare memory on vs off

Ask the identical question with an empty graph. The only difference is the JSON store.

In [14]:
# q = "What project am I working on, and what is it about?"

empty = KG()
system = "You are a helpful assistant. Known facts:\n" + ("\n".join(empty.lines(q)) or "(none)")
print("Model (no memory):", chat([{"role": "system", "content": system}, {"role": "user", "content": q}]))


Model (no memory): You haven't shared any information about the project you're working on yet. Could you provide some details so I can assist you better?


## Appendix A - Run fully local with Ollama

The same loop runs offline: point an OpenAI-compatible client at a local Ollama server with an open-weight model. Requires `ollama pull ministral-3:8b` and `pip install openai`.

In [ ]:
# !pip install -q openai
# from openai import OpenAI
# local = OpenAI(base_url="http://localhost:11434/v1", api_key="local")
# def chat_local(messages, temperature=0.7):
#     r = local.chat.completions.create(model="ministral-3:8b", messages=messages, temperature=temperature)
#     return r.choices[0].message.content.strip()
# Swap `chat` for `chat_local` above to run everything on-device.


## Appendix B - Narrate output with Voxtral TTS

If you have an enrolled voice, a reply or summary can be spoken aloud. Requires a Mistral API key with audio access; create a voice with `client.audio.voices.create(...)`.

In [ ]:
# voice_id = "<your-enrolled-voice-id>"
# resp = client.audio.speech.complete(model="voxtral-mini-tts-2603", input="Summary: ...", voice_id=voice_id)
# save the returned audio bytes to an .mp3 and play it


## Limitations

- Keyword + 2-hop retrieval does not match paraphrases; for semantic recall, add embeddings at the retrieval step.
- Extraction quality depends on the model; audit the stored graph and keep a manual edit path.
- Single-user, single-process: no locking or access control — scope users by separate files.

## Summary

This cookbook showed how to attach persistent memory to a chat model: after each message, extract facts as JSON triples and merge them into a knowledge graph stored as plain JSON; before each call, retrieve the relevant 2-hop subgraph into the system prompt; and consolidate contradictions offline.

**What this cookbook covers**

- Demonstrating that a chat call keeps no state between requests
- Extracting structured triples with JSON mode at temperature 0
- Storing and retrieving a knowledge graph in plain JSON (keyword + 2-hop)
- Answering multi-hop questions in a fresh session from the graph alone
- Offline consolidation that fixes conflicting edges and writes a summary
- A memory on/off comparison with the same model

**Mistral features used**

- Chat Completions API with `response_format={"type": "json_object"}` (structured output)
- Voxtral TTS (`voxtral-mini-tts-2603`) in Appendix B

**Other services**

- Ollama with an open-weight model (Appendix A)

[View the structured output documentation](https://docs.mistral.ai/capabilities/structured_output/)
